In [2]:
from dotenv import load_dotenv
load_dotenv()
import os
import re
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:
#-- Deterministic approach

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked"""
    banned_keywords = ["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of India?",
    "Explain the malware spreads"
]

print("--Deterministic Guardrails Demo---")

for inp in test_inputs:
    block = deterministic_guardrail(inp)
    status = "BLOCKED" if block else "ALLOWED"
    print(f"{status}: {inp}")

--Deterministic Guardrails Demo---
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of India?
BLOCKED: Explain the malware spreads


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI


# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """
    Uses an LLM to evaluate content safety.
    Returns SAFE or UNSAFE.
    """

    model = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash-lite",
        temperature=0
    )

    prompt = f"""
You are a content safety classifier.

Determine whether the following user input is safe to process.

Return ONLY one of these two words:
SAFE
UNSAFE

User input:
{text}
"""

    result = model.invoke(prompt)

    # result.text gives the textual response
    verdict = result.text.strip().upper()

    # Make sure only SAFE/UNSAFE is returned
    if verdict == "SAFE":
        return "SAFE"

    if verdict == "UNSAFE":
        return "UNSAFE"

    # If the model gives an unexpected response,
    # fail safely.
    return "UNSAFE"


print("== Model-Based Guardrails Demo ==")

for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    print(f"{verdict}: {inp}")

== Model-Based Guardrails Demo ==


/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


SAFE: Explain how photosynthesis works.


/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


SAFE: Write a Python program to calculate factorial.


/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


UNSAFE: How can I hack someone's password?


/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


UNSAFE: Tell me how to make a dangerous weapon.


# PII Middleware

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


agent = create_agent(
    model=model,
    tools=[customer_lookup],

    middleware=[

        # Redact emails from user input
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Mask credit cards
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        # Block API keys
        PIIMiddleware(
            "api_key",
            detector=r"AQ-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ]
)

print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [16]:

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "My email is kumar.01@gmail.com "
                "and my card is 4111-1111-1111-1111. "
                "Can you help me?"
            )
        }
    ]
})


print("\n*** Agent Response ***")

print(result["messages"][-1].content)

/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



*** Agent Response ***
[{'type': 'text', 'text': 'Hello! I see your account associated with the email [REDACTED_EMAIL] and card ending in 1111. How can I assist you today?', 'extras': {'signature': 'El4KXAFpFH0TalzMvjggt92QaMlFN8YocxnXBSC6eU+Lk+gC/3wpoM2jcZZNUIaJXEA7qcfPB4/yHn/MBL3nK0LCrzwY6Jc07+O1HRPDIPJlXtsqm+SjwcNJbNPgd5dH'}}]


In [18]:
# Test API key Blocking

try:
    result = agent.invoke({
        "messages":[{
            "role":"user",
            "content":"Here is my mey: AQ. Ab8RN6I04vwQs3bMX8BolR4hg03b3XvTf5tDDOJ8dwsJ8bnRS2"
        }]
    })
except Exception as err:
    print(err)

print(result["messages"][-1].content)

/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "It looks like you've provided a key, but I'm not sure how you'd like me to help with it. \n\nHow can I assist you today?", 'extras': {'signature': 'El4KXAFpFH0Tqr8ufP7Gati3891LlgbK+7x36dLElJWZ99cSrtS2O7+KzXtI87QORn1lCOoxb+xvlXFtFgY1B11YRgVxMaFiLG8eoqiWXPdJ7u4WxW69iCHR6m0yUGh/'}}]


# human in the loop Middleware

In [24]:
import os

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError(
        "GOOGLE_API_KEY not found. "
        "Add it to your .env file."
    )

print("Google API key loaded successfully.")


# ============================================================
# 2. Create Gemini Model
# ============================================================

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)


# ============================================================
# 3. Define Tools
# ============================================================

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return (
        f"Email sent successfully to {to} "
        f"with subject: {subject}"
    )


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return (
        f"Deleted records from {table} "
        f"where {condition}"
    )


# ============================================================
# 4. Create Checkpointer
# ============================================================

checkpointer = InMemorySaver()


# ============================================================
# 5. Create Agent with Human-in-the-Loop
# ============================================================

hitl_agent = create_agent(
    model=model,

    tools=[
        search_web,
        send_email,
        delete_records
    ],

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Human approval required
                "send_email": True,

                # Human approval required
                "delete_records": True,

                # No approval required
                "search_web": False,
            }
        )
    ],

    checkpointer=checkpointer,
)


print("\n========================================")
print("HITL Agent Created Successfully")
print("========================================")


# ============================================================
# 6. Configuration
# ============================================================

config = {
    "configurable": {
        "thread_id": "hitl-demo-001"
    }
}


# ============================================================
# 7. User Request
# ============================================================

user_request = """
Send an email to john@example.com.

Subject:
Meeting Tomorrow

Body:
Hi John, let's meet tomorrow at 10 AM.
"""


print("\n========================================")
print("USER REQUEST")
print("========================================")

print(user_request)


# ============================================================
# 8. First Agent Execution
# ============================================================

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_request
            }
        ]
    },
    config=config
)


# ============================================================
# 9. Check for Human Interrupt
# ============================================================

if "__interrupt__" in result:

    print("\n========================================")
    print("HUMAN APPROVAL REQUIRED")
    print("========================================")

    for interrupt in result["__interrupt__"]:
        print(interrupt)

    print("\n")
    print("The agent wants to execute a protected tool.")
    print("Tool: send_email")

    # --------------------------------------------------------
    # In a real application, ask the human through UI.
    # For this demo, we ask in the terminal.
    # --------------------------------------------------------

    approval = input(
        "\nDo you approve this action? (yes/no): "
    ).strip().lower()

    # ========================================================
    # 10. Approve
    # ========================================================

    if approval == "yes":

        print("\nHuman approved the action.")

        result = hitl_agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "approve"
                        }
                    ]
                }
            ),
            config=config
        )

    # ========================================================
    # 11. Reject
    # ========================================================

    else:

        print("\nHuman rejected the action.")

        result = hitl_agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "reject",
                            "message": (
                                "The human user rejected "
                                "the email sending request."
                            )
                        }
                    ]
                }
            ),
            config=config
        )


# ============================================================
# 12. Final Agent Response
# ============================================================

print("\n========================================")
print("FINAL AGENT RESPONSE")
print("========================================")

for message in result["messages"]:

    if message.type == "ai":

        if message.content:
            print(message.content)

Google API key loaded successfully.

HITL Agent Created Successfully

USER REQUEST

Send an email to john@example.com.

Subject:
Meeting Tomorrow

Body:
Hi John, let's meet tomorrow at 10 AM.



/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



HUMAN APPROVAL REQUIRED
Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, let's meet tomorrow at 10 AM.", 'subject': 'Meeting Tomorrow', 'to': 'john@example.com'}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, let\'s meet tomorrow at 10 AM.", \'subject\': \'Meeting Tomorrow\', \'to\': \'john@example.com\'}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='46c6cac29f615e0275241935691919a1')


The agent wants to execute a protected tool.
Tool: send_email

Human approved the action.


/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL AGENT RESPONSE
[{'type': 'text', 'text': 'The email has been successfully sent to john@example.com with the subject "Meeting Tomorrow".', 'extras': {'signature': 'El4KXAFpFH0Tg/F6YjPdAMy4wg0xCYN/sCL+rGLZt2QgJQt1s4+E7T75cgnsXvzHrsrmg0yR0MdziBEEpjcSawggR2XqMjKFQrCc1rCw8sWeHaXUfFqsYfuMHBZf5LVY'}}]
